# ptsrv.render

> Scaled raster plans, sections, elevations, and world-file output.

In [ ]:
#| default_exp render

In [ ]:
#| export
"""Orthographic raster rendering of plans and sections from a point cloud.

All renders share one core: project points into a 2-D image plane, keep the
point nearest the viewer per pixel (z-buffer), then optionally

  * fill small gaps (nearest-neighbour within a radius) so sparse LiDAR
    slices read as surfaces,
  * draw the "cut face" (points within a thin band at the cut plane) in a
    solid colour on top, which gives the heavy wall lines of an
    architectural plan/section,
  * write a world file / DPI so the PNG is at true scale in CAD and print.

Coordinate conventions
  Plan:    image x = world +X (east), image y (down) = world -Y (north up).
  Section: image x = along the section line A->B, image y (down) = -Z.
           The viewer stands on the side of the line given by `side`
           (left/right when walking A->B) and looks toward the other side;
           points between the line and `depth` metres beyond it are drawn.
"""
from __future__ import annotations

In [ ]:
#| export
import io
import json
import math
import zipfile
from dataclasses import dataclass, field

In [ ]:
#| export
import numpy as np
from PIL import Image

In [ ]:
#| export
from .cloud import Cloud

In [ ]:
#| export
Style = str  # "color" | "depth" | "density" | "hybrid"

In [ ]:
#| export
@dataclass
class RenderResult:
    image: Image.Image
    pixel_size: float            # metres per pixel
    origin: tuple[float, float]  # world coords of the CENTRE of the top-left pixel (u, v) in the view frame
    frame: dict                  # description of the view frame (for JSON / world file)
    stats: dict = field(default_factory=dict)

    # -- georeferencing --------------------------------------------------
    def world_file(self) -> str:
        """ESRI world file (.pgw).  Valid for plans (image axes == world X / -Y).

        For sections the "world" is the 2-D section frame (u along the line,
        v = elevation); still useful for placing the image at scale in CAD.
        """
        px = self.pixel_size
        ox, oy = self.origin
        return f"{px:.8f}\n0.0\n0.0\n{-px:.8f}\n{ox:.6f}\n{oy:.6f}\n"

    def png_bytes(self, print_scale: float | None = None) -> bytes:
        """Encode to PNG.  If print_scale (e.g. 50 for 1:50) is given, embed DPI so
        the image prints at that scale."""
        buf = io.BytesIO()
        kw = {}
        if print_scale:
            dpi = 0.0254 * print_scale / self.pixel_size
            kw["dpi"] = (dpi, dpi)
        self.image.save(buf, format="PNG", optimize=False, compress_level=6, **kw)
        return buf.getvalue()

    def metadata(self) -> dict:
        w, h = self.image.size
        return {
            "width_px": w, "height_px": h,
            "pixel_size_m": self.pixel_size,
            "origin_top_left_centre": list(self.origin),
            "frame": self.frame,
            "stats": self.stats,
        }

    def zip_bytes(self, basename: str, print_scale: float | None = None) -> bytes:
        buf = io.BytesIO()
        with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
            z.writestr(f"{basename}.png", self.png_bytes(print_scale))
            z.writestr(f"{basename}.pgw", self.world_file())
            z.writestr(f"{basename}.json", json.dumps(self.metadata(), indent=2))
        return buf.getvalue()

In [ ]:
#| export
def _zbuffer_raster(u: np.ndarray, v: np.ndarray, depth: np.ndarray, rgb: np.ndarray,
                    u0: float, v1: float, w: int, h: int, px: float):
    """Project (u,v) points to an image of w x h pixels.

    u0 = world u of the left image edge, v1 = world v of the top image edge.
    Returns (color (h,w,3) uint8, depth (h,w) float32 with inf where empty,
             count (h,w) int32).
    Nearest (smallest depth) point wins per pixel.
    """
    ix = np.floor((u - u0) / px).astype(np.int64)
    iy = np.floor((v1 - v) / px).astype(np.int64)
    ok = (ix >= 0) & (ix < w) & (iy >= 0) & (iy < h)
    ix, iy, depth, rgb = ix[ok], iy[ok], depth[ok], rgb[ok]

    color = np.zeros((h, w, 3), np.uint8)
    dbuf = np.full((h, w), np.inf, np.float32)
    count = np.zeros((h, w), np.int32)
    if len(ix) == 0:
        return color, dbuf, count

    np.add.at(count, (iy, ix), 1)
    # sort far -> near so that the last write per pixel is the nearest point
    order = np.argsort(-depth, kind="stable")
    iy, ix, depth, rgb = iy[order], ix[order], depth[order], rgb[order]
    color[iy, ix] = rgb
    dbuf[iy, ix] = depth
    return color, dbuf, count

In [ ]:
#| export
def _fill_gaps(color: np.ndarray, dbuf: np.ndarray, radius_px: int):
    """Fill empty pixels from the nearest filled pixel within radius_px."""
    if radius_px <= 0:
        return color, dbuf
    from scipy import ndimage
    filled = np.isfinite(dbuf)
    if filled.all() or not filled.any():
        return color, dbuf
    h, w = dbuf.shape
    # Pack (quantised depth, pixel index) into one int64 so a separable
    # minimum filter returns the key of the NEAREST filled pixel in the window.
    # This is ~5x faster than a full Euclidean distance transform with indices.
    dmax = float(dbuf[filled].max()) or 1.0
    dq = np.zeros((h, w), np.int64)
    dq[filled] = np.clip(dbuf[filled] / dmax * (2**30 - 1), 0, 2**30 - 1).astype(np.int64)
    idx = np.arange(h * w, dtype=np.int64).reshape(h, w)
    key = np.where(filled, (dq << 32) | idx, np.iinfo(np.int64).max)
    kmin = ndimage.minimum_filter(key, size=2 * radius_px + 1, mode="nearest")
    src = (kmin & 0xFFFFFFFF)
    fill = (~filled) & (kmin != np.iinfo(np.int64).max)
    color = color.copy()
    dbuf = dbuf.copy()
    flat_c = color.reshape(-1, 3)
    flat_d = dbuf.reshape(-1)
    color[fill] = flat_c[src[fill]]
    dbuf[fill] = flat_d[src[fill]]
    return color, dbuf

In [ ]:
#| export
def _shade(color: np.ndarray, dbuf: np.ndarray, count: np.ndarray, style: Style,
           depth_range: float, background: tuple[int, int, int]):
    """Turn raw buffers into an RGB image according to style."""
    filled = np.isfinite(dbuf)
    h, w = dbuf.shape
    out = np.empty((h, w, 3), np.uint8)
    out[:] = background

    if style == "color":
        out[filled] = color[filled]
    elif style == "depth":
        # near = dark, far = light  (reads like a shaded drawing)
        d = np.clip(dbuf / max(depth_range, 1e-6), 0, 1)
        g = (40 + 200 * d).astype(np.uint8)
        out[filled] = np.stack([g, g, g], -1)[filled]
    elif style == "density":
        c = np.log1p(count.astype(np.float32))
        c = c / max(c.max(), 1e-6)
        g = (255 - 215 * c).astype(np.uint8)
        out[count > 0] = np.stack([g, g, g], -1)[count > 0]
    elif style == "hybrid":
        # colour modulated by depth: nearer surfaces slightly darker, far lighter
        d = np.clip(dbuf / max(depth_range, 1e-6), 0, 1)
        f = (0.55 + 0.45 * d)[..., None]
        mixed = (color.astype(np.float32) * f + 255 * (1 - f) * 0.35).clip(0, 255).astype(np.uint8)
        out[filled] = mixed[filled]
    else:
        raise ValueError(f"unknown style {style!r}")
    return out

In [ ]:
#| export
def _draw_cut(out: np.ndarray, u: np.ndarray, v: np.ndarray, u0: float, v1: float, px: float,
              colour: tuple[int, int, int], thicken_px: int):
    w, h = out.shape[1], out.shape[0]
    ix = np.floor((u - u0) / px).astype(np.int64)
    iy = np.floor((v1 - v) / px).astype(np.int64)
    ok = (ix >= 0) & (ix < w) & (iy >= 0) & (iy < h)
    if not ok.any():
        return out
    mask = np.zeros((h, w), bool)
    mask[iy[ok], ix[ok]] = True
    if thicken_px > 0:
        from scipy import ndimage
        mask = ndimage.binary_dilation(mask, iterations=thicken_px)
    out[mask] = colour
    return out

In [ ]:
#| export
def _render_view(u, v, depth, rgb, cut_u, cut_v, *, px, u_range, v_range, style, fill_radius_m,
                 cut_colour, cut_thicken_m, background, depth_range, max_pixels, frame):
    u_min, u_max = u_range
    v_min, v_max = v_range
    w = int(math.ceil((u_max - u_min) / px))
    h = int(math.ceil((v_max - v_min) / px))
    if w <= 0 or h <= 0:
        raise ValueError("empty view extent")
    if w * h > max_pixels:
        raise ValueError(f"image would be {w}x{h} = {w*h/1e6:.0f} Mpx; raise pixel size or crop (limit {max_pixels/1e6:.0f} Mpx)")

    color, dbuf, count = _zbuffer_raster(u, v, depth, rgb, u_min, v_max, w, h, px)
    color, dbuf = _fill_gaps(color, dbuf, int(round(fill_radius_m / px)))
    out = _shade(color, dbuf, count, style, depth_range, background)
    if cut_u is not None and len(cut_u):
        out = _draw_cut(out, cut_u, cut_v, u_min, v_max, px, cut_colour, int(round(cut_thicken_m / px)))

    img = Image.fromarray(out, "RGB")
    origin = (u_min + px / 2, v_max - px / 2)
    stats = {"points_in_view": int(len(u)), "filled_fraction": float(np.isfinite(dbuf).mean())}
    return RenderResult(image=img, pixel_size=px, origin=origin, frame=frame, stats=stats)

In [ ]:
#| export
def render_plan(cloud: Cloud, *, cut_height: float = 1.2, level: float | None = None,
                below: float | None = None, look: str = "down",
                pixel_size: float = 0.005, style: Style = "hybrid",
                bounds: tuple[float, float, float, float] | None = None,
                margin: float = 0.25, fill_radius: float = 0.02,
                cut_band: float = 0.03, cut_colour=(0, 0, 0), cut_thicken: float = 0.0,
                background=(255, 255, 255), max_pixels: int = 60_000_000) -> RenderResult:
    """Render a floor plan.

    cut_height  height of the cut plane above `level` (m).  Standard is 1.0–1.2.
    level       world Z of the floor; default = detected floor.
    below       how far below the cut plane to keep points (default: down to
                0.05 m below floor).  Use to hide lower storeys.
    look        "down" (floor plan) or "up" (reflected ceiling plan; then
                cut_height is measured up from level and points ABOVE the cut are drawn).
    bounds      (xmin, ymin, xmax, ymax) crop in world coords; default = whole cloud.
    cut_band    thickness (m) of the slab at the cut plane drawn in cut_colour.
                0 disables.
    style       color | depth | density | hybrid
    """
    xyz, rgb, info = cloud.xyz, cloud.rgb, cloud.info
    level = info.floor_z if level is None else level
    cut_z = level + cut_height
    z = xyz[:, 2]

    if look == "down":
        lo = (level - 0.05) if below is None else (cut_z - below)
        sel = (z <= cut_z) & (z >= lo)
        depth = cut_z - z[sel]
        depth_range = cut_z - lo
    elif look == "up":
        hi = (info.ceiling_z + 0.05) if below is None else (cut_z + below)
        sel = (z >= cut_z) & (z <= hi)
        depth = z[sel] - cut_z
        depth_range = hi - cut_z
    else:
        raise ValueError("look must be 'down' or 'up'")

    p = xyz[sel]
    c = rgb[sel]
    if bounds is None:
        bounds = (info.xmin - margin, info.ymin - margin, info.xmax + margin, info.ymax + margin)
    xmin, ymin, xmax, ymax = bounds

    cut_sel = np.abs(z - cut_z) <= cut_band / 2 if cut_band > 0 else np.zeros(len(z), bool)
    frame = {"type": "plan", "look": look, "cut_z": float(cut_z), "level_z": float(level),
             "cut_height": float(cut_height), "bounds": [float(b) for b in bounds],
             "rotation_applied_deg": info.rotation_deg}
    return _render_view(p[:, 0], p[:, 1], depth, c, xyz[cut_sel, 0], xyz[cut_sel, 1],
                        px=pixel_size, u_range=(xmin, xmax), v_range=(ymin, ymax), style=style,
                        fill_radius_m=fill_radius, cut_colour=cut_colour, cut_thicken_m=cut_thicken,
                        background=background, depth_range=depth_range, max_pixels=max_pixels,
                        frame=frame)

In [ ]:
#| export
def render_section(cloud: Cloud, *, a: tuple[float, float], b: tuple[float, float],
                   depth: float = 1.0, side: str = "left",
                   zmin: float | None = None, zmax: float | None = None,
                   pixel_size: float = 0.005, style: Style = "hybrid",
                   margin: float = 0.15, fill_radius: float = 0.02,
                   cut_band: float = 0.03, cut_colour=(0, 0, 0), cut_thicken: float = 0.0,
                   background=(255, 255, 255), max_pixels: int = 60_000_000) -> RenderResult:
    """Render a vertical section / elevation.

    a, b     endpoints of the section line in world XY (m).
    side     which side of the line the viewer stands on when walking a -> b:
             "left" or "right".  The view looks across the line and shows
             everything from the line to `depth` metres on the far side.
    depth    how far beyond the cut plane to draw (m).  Small (0.2) = pure
             section slice; large = section + elevation of what is behind.
    zmin/zmax vertical crop; default = cloud floor-0.1 .. ceiling+0.1.

    The image is a true (un-mirrored) view from the viewer's position:
    side="right" -> a is on the image's left, b on the right;
    side="left"  -> b is on the image's left, a on the right.
    `frame["image_left_is"]` in the result records which.
    """
    xyz, rgb, info = cloud.xyz, cloud.rgb, cloud.info
    ax, ay = a
    bx, by = b
    dx, dy = bx - ax, by - ay
    L = math.hypot(dx, dy)
    if L < 1e-6:
        raise ValueError("section endpoints coincide")
    tu = np.array([dx / L, dy / L], np.float32)          # along the line
    n_left = np.array([-tu[1], tu[0]], np.float32)       # left normal walking a->b
    if side == "left":
        view = -n_left        # viewer on the left looks toward the right side
    elif side == "right":
        view = n_left
    else:
        raise ValueError("side must be 'left' or 'right'")

    rel = xyz[:, :2] - np.array([ax, ay], np.float32)
    u = rel @ tu            # position along line
    d = rel @ view          # distance beyond the cut plane, positive = away from viewer
    z = xyz[:, 2]
    zlo = (info.floor_z - 0.1) if zmin is None else zmin
    zhi = (info.ceiling_z + 0.1) if zmax is None else zmax

    sel = (d >= 0) & (d <= depth) & (u >= -margin) & (u <= L + margin) & (z >= zlo) & (z <= zhi)
    cut_sel = (np.abs(d) <= cut_band / 2) & (u >= -margin) & (u <= L + margin) & (z >= zlo) & (z <= zhi) \
        if cut_band > 0 else np.zeros(len(z), bool)

    # Un-mirror: when the viewer stands on the left, walking a->b means b is on
    # the viewer's left and a on the right in a true elevation.  Flip u so the
    # image is what you would actually see from the viewpoint.
    if side == "left":
        u_img = L - u
    else:
        u_img = u

    frame = {"type": "section", "a": [float(ax), float(ay)], "b": [float(bx), float(by)],
             "side": side, "depth": float(depth), "zmin": float(zlo), "zmax": float(zhi),
             "length": float(L), "image_left_is": "b" if side == "left" else "a",
             "rotation_applied_deg": info.rotation_deg}
    return _render_view(u_img[sel], z[sel], d[sel], rgb[sel], u_img[cut_sel], z[cut_sel],
                        px=pixel_size, u_range=(-margin, L + margin), v_range=(zlo, zhi), style=style,
                        fill_radius_m=fill_radius, cut_colour=cut_colour, cut_thicken_m=cut_thicken,
                        background=background, depth_range=depth, max_pixels=max_pixels, frame=frame)

In [ ]:
#| export
def render_elevation(cloud: Cloud, *, a, b, side="left", depth=50.0, **kw) -> RenderResult:
    """Elevation = section with a large depth and no cut band."""
    kw.setdefault("cut_band", 0.0)
    return render_section(cloud, a=a, b=b, side=side, depth=depth, **kw)